In [37]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk
from sklearn.cluster import KMeans

nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ag331\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [38]:
csv_path = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/processed_dataset.csv"
image_folder = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized"

df = pd.read_csv(csv_path)

In [39]:
import re
from collections import Counter

# ----------------------------
# ✅ Tokenizer and Vocabulary
# ----------------------------
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)     # remove punctuation
    text = re.sub(r"\d+", "", text)         # remove numbers
    return text.strip().split()

def build_vocab(df):
    # Merge text columns for tokenization
    df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('')
    tokenized_reports = [tokenize(r) for r in df['report']]

    # Collect vocabulary
    special_tokens = ["<pad>", "<start>", "<end>", "<unk>"]
    all_tokens = [w for tokens in tokenized_reports for w in tokens]
    vocab = special_tokens + [w for w, c in Counter(all_tokens).items() if c >= 2]

    # Create mappings
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}

    return word2idx, idx2word


In [40]:
import nltk
nltk.download('punkt')  # <- add this explicitly before using word_tokenize


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ag331\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [41]:


# Corrected: Combine impression, findings, and Problems
df["report"] = df[["impression", "findings", "Problems"]].fillna("").agg(" ".join, axis=1)

image_paths = [os.path.join(image_folder, fname) for fname in df["filename"]]
reports = list(df["report"])
labels = list(df["MeSH"])  # or use another label column as appropriate



transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])


In [42]:

# ---------------------- Datasets ----------------------
class ChestXrayDataset(Dataset):
    def __init__(self, image_paths, reports, labels, transform=None):
        self.image_paths = image_paths
        self.reports = reports
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.reports[idx], self.labels[idx]

class RefinedLabelDataset(Dataset):
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [43]:
# ---------------------- Models ----------------------
class CNNFeatureExtractor(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.googlenet(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

class ReportRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size + embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        embeddings = self.embedding(captions)
        features = features.unsqueeze(1).expand(-1, embeddings.size(1), -1)
        combined = torch.cat((features, embeddings), dim=2)
        outputs, _ = self.gru(combined)
        return self.linear(outputs)

In [44]:
dataset = ChestXrayDataset(image_paths, reports, labels, transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# ---------------------- Training Functions ----------------------
def retrain_cnn(image_paths, new_labels, transform, model, device, epochs=10):
    dataset = RefinedLabelDataset(image_paths, new_labels, transform)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
    model.fc = nn.Linear(model.fc.in_features, len(set(new_labels)))
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(model(imgs), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs} | CNN Loss: {loss.item():.4f}")
    return model


In [45]:
def retrain_rnn(rnn_model, cnn_model, dataloader, tokenizer, device, vocab_size, epochs=10):
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    optimizer = torch.optim.Adam(rnn_model.parameters(), lr=1e-4)
    rnn_model.train()
    cnn_model.eval()
    for epoch in range(epochs):
        for images, reports, _ in dataloader:
            images = images.to(device)
            cnn_feats = cnn_model(images).unsqueeze(1)
            tokenized = [tokenizer.encode(r, max_length=5) for r in reports]
            tokens = torch.tensor(tokenized).to(device)
            inputs = tokens[:, :-1]
            targets = tokens[:, 1:]
            embeddings = rnn_model.embedding(inputs)
            features = cnn_feats.expand(-1, embeddings.size(1), -1)
            combined = torch.cat((features, embeddings), dim=2)
            outputs, _ = rnn_model.gru(combined)
            logits = rnn_model.linear(outputs)
            loss = criterion(logits.view(-1, vocab_size), targets.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{epochs} | RNN Loss: {loss.item():.4f}")
